# 數據儲存與管理

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 區分結構化、半結構化與非結構化資料。
2. 理解分散式儲存中的切分、副本與容錯概念。
3. 用小型資料模擬資料湖的 schema-on-read 思維。
4. 比較分區查詢、欄位裁剪與壓縮對儲存與查詢效率的影響。
5. 依照資料型態與使用情境，選擇合適的儲存管理策略。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節示範所需的 Python 套件，並建立可重現的隨機種子。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter, defaultdict

np.random.seed(42)
print('環境設定完成')


## 核心概念說明

在大數據與 AI 應用中，資料儲存與管理的重點不只是把資料放起來，而是要讓資料能穩定、快速、安全地被取用。

常見資料型態包含：

- **結構化資料**：有固定欄位與資料型別，例如訂單表、會員資料表。
- **半結構化資料**：有標記或階層，但欄位不一定固定，例如 JSON、XML。
- **非結構化資料**：沒有明確表格結構，例如圖片、影片、PDF、客服對話。

常見儲存設計包含：

- **RDBMS**：適合結構穩定、需要 SQL 與交易一致性的資料。
- **文件型 NoSQL**：適合欄位彈性高的 JSON 類資料。
- **物件儲存或 HDFS**：適合大量檔案與非結構化資料。
- **資料湖**：保留原始資料，讀取時再套用結構，也就是 schema-on-read。


In [ ]:
# ── 示範：資料型態分類 ───────────────────────────────
# 這段程式碼用簡化規則判斷資料樣本屬於結構化、半結構化或非結構化資料。

import pandas as pd
import re

samples = [
    {'name': 'orders_table', 'sample': 'order_id,customer_id,amount,order_date'},
    {'name': 'api_payload', 'sample': '{"sensor_id":"A01", "temp":26.5, "ts":"2026-06-10"}'},
    {'name': 'product_image', 'sample': 'images/product_001.jpg'},
    {'name': 'customer_note', 'sample': '客戶反映配送遲，希望客服回覆'},
    {'name': 'log_record', 'sample': '{"level":"WARN", "service":"payment", "retry":3}'},
]

def classify_data_type(text):
    text = str(text).strip()
    if text.startswith('{') and text.endswith('}'):
        return '半結構化資料'
    if re.search(r'\.(jpg|png|mp4|pdf|wav)$', text, re.IGNORECASE):
        return '非結構化資料'
    if ',' in text and len(text.split(',')) >= 3:
        return '結構化資料'
    return '非結構化資料'

result = pd.DataFrame(samples)
result['data_type'] = result['sample'].apply(classify_data_type)
print(result)


## 分散式儲存與資料湖

分散式儲存會把資料切成多個區塊，分散到不同節點，並透過副本或冗餘機制降低單一節點故障造成的風險。

重要觀念：

- **切分 Data Chunks**：把大資料拆成小區塊。
- **副本 Replication**：同一份資料放在多個節點。
- **容錯 Fault Tolerance**：節點失效時仍可讀到資料。
- **資料本地性 Data Locality**：盡量讓運算靠近資料，降低傳輸成本。

資料湖則常用於整合多來源、多格式的原始資料。它不要求資料在寫入時就完全結構化，而是在讀取分析時再套用欄位結構。


In [ ]:
# ── 示範：分散式切分與副本 ─────────────────────────────
# 這段程式碼模擬把資料切成區塊，並建立副本到多個節點，觀察節點故障後資料是否仍可讀取。

import numpy as np
import pandas as pd
from collections import defaultdict

np.random.seed(7)
records = [f'record_{i:02d}' for i in range(12)]
nodes = ['node_A', 'node_B', 'node_C', 'node_D']
replication_factor = 2

placement = defaultdict(list)
for record in records:
    selected_nodes = np.random.choice(nodes, size=replication_factor, replace=False)
    for node in selected_nodes:
        placement[record].append(node)

placement_df = pd.DataFrame([
    {'record': record, 'replicas': ', '.join(replica_nodes)}
    for record, replica_nodes in placement.items()
])
print('資料副本配置：')
print(placement_df)

failed_node = 'node_B'
available = []
for record, replica_nodes in placement.items():
    available.append({
        'record': record,
        'readable_after_failure': any(node != failed_node for node in replica_nodes)
    })

availability_df = pd.DataFrame(available)
print('\n節點 node_B 故障後可讀取比例：')
print(availability_df['readable_after_failure'].mean())


## 儲存效能優化與查詢管理

資料量變大後，查詢慢通常不是因為 Python 或 SQL 本身，而是因為掃描了太多不必要的資料。

常見策略：

- **分區儲存 Partitioning**：依月份、地區、設備類型等欄位切分資料，查詢時只掃描相關分區。
- **欄位裁剪 Column Pruning**：只讀取需要的欄位，減少 I/O。
- **壓縮 Compression**：降低儲存空間與傳輸量，常搭配 Parquet、ORC 等列式格式。
- **儲存分層 Storage Tiering**：熱資料放高速儲存，冷資料放低成本儲存。

以下用小型 pandas 資料模擬這些概念。


In [ ]:
# ── 實際應用：分區查詢、欄位裁剪與壓縮估算 ─────────────────────
# 這段程式碼建立模擬交易資料，比較全表掃描與分區查詢的掃描量，並估算欄位裁剪與壓縮帶來的資料量差異。

import numpy as np
import pandas as pd

np.random.seed(10)
n = 5000
df = pd.DataFrame({
    'month': np.random.choice(['2026-01', '2026-02', '2026-03', '2026-04'], size=n, p=[0.2, 0.25, 0.25, 0.3]),
    'region': np.random.choice(['北部', '中部', '南部', '東部'], size=n),
    'customer_id': np.random.randint(1000, 2000, size=n),
    'amount': np.random.gamma(shape=2.0, scale=600.0, size=n).round(0),
    'event_text': np.random.choice(['click', 'view', 'purchase', 'refund'], size=n)
})

query_month = '2026-03'
full_scan_rows = len(df)
partition_scan_rows = len(df[df['month'] == query_month])
scan_reduction = 1 - partition_scan_rows / full_scan_rows

all_columns_memory = df.memory_usage(deep=True).sum()
needed_columns = ['month', 'amount']
pruned_memory = df[needed_columns].memory_usage(deep=True).sum()
column_reduction = 1 - pruned_memory / all_columns_memory

compression_ratio = 0.35
estimated_compressed_size = int(pruned_memory * compression_ratio)

summary = pd.DataFrame({
    'metric': ['全表掃描列數', '分區掃描列數', '分區減少掃描比例', '全欄位記憶體bytes', '欄位裁剪後bytes', '欄位裁剪減少比例', '估計壓縮後bytes'],
    'value': [full_scan_rows, partition_scan_rows, round(scan_reduction, 3), all_columns_memory, pruned_memory, round(column_reduction, 3), estimated_compressed_size]
})
print(summary)

monthly_amount = df[df['month'] == query_month].groupby('region')['amount'].sum().sort_values(ascending=False)
print('\n2026-03 各區銷售額：')
print(monthly_amount)


In [ ]:
# ── 🧪 自我測驗 ──────────────────────────────────
# 請完成下方 TODO 填空，實作分區查詢與欄位裁剪，並確認輸出符合 Expected 註解。

import pandas as pd

logs = pd.DataFrame({
    'month': ['2026-01', '2026-01', '2026-02', '2026-02', '2026-03'],
    'service': ['api', 'web', 'api', 'batch', 'web'],
    'latency_ms': [120, 95, 180, 240, 110],
    'bytes_scanned': [1500, 1200, 2100, 3200, 900],
    'status': ['ok', 'ok', 'warn', 'ok', 'ok']
})

target_month = '2026-02'
needed_columns = ['service', 'latency_ms']

# TODO 1: 用 target_month 進行分區查詢，只保留 2026-02 的紀錄。
partitioned_logs = logs[logs['month'] == target_month]

# TODO 2: 進行欄位裁剪，只保留 service 與 latency_ms 欄位。
pruned_logs = partitioned_logs[needed_columns]

# TODO 3: 計算分區查詢後的平均 latency_ms。
avg_latency = pruned_logs['latency_ms'].mean()

print('分區查詢列數:', len(partitioned_logs))
print('\n欄位裁剪結果:')
print(pruned_logs)
print('\n2026-02 平均延遲:', avg_latency)

# Expected:
# 分區查詢列數: 2
#
# 欄位裁剪結果:
#   service  latency_ms
# 2     api         180
# 3   batch         240
#
# 2026-02 平均延遲: 210.0
